In [ ]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 베이지안 최적화 라이브러리 설치
!pip install bayesian-optimization
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.9 MB/s eta 0:00:00


### DT에 사용할 파라이터 예시
    random_state=42.
    class_weight='balanced',     # 낙상 클래스 가중치 자동 조정
    max_depth=10,                # 과적합 방지
    min_samples_split=10,        # 너무 잘게 분할 방지
    min_samples_leaf=5,          # 리프 노드 최소 샘플 보장
    criterion='gini',        
    ccp_alpha=0.01,              # 가지치기로 일반화 성능 향상


In [ ]:
# 기본 라이브러리 할당
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd

# 데이터 분할 및 스케링 라이브러리 할당
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 모델 라이브러리 할당
from sklearn.tree import DecisionTreeClassifier
import optuna

# 모델 평가 라이브러리 할당
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)
from sklearn.model_selection import cross_val_score

In [ ]:
df_path = "/content/drive/MyDrive/OnSafe/modeling_final.csv"
df = pd.read_csv(df_path)

In [ ]:
# 제외할 키워드 목록
exclude_keywords = ['video', 'file_id', 'frame', 'timestamp']

# 키워드가 하나라도 포함되면 제외
drop_col = [col for col in df.columns if not any(keyword in col.lower() for keyword in exclude_keywords)]

# 컬럼 선택
df = df[drop_col]

df

,neck_angle,neck_angular_velocity,neck_angular_acceleration,shoulder_balance_angle,shoulder_balance_angular_velocity,shoulder_balance_angular_acceleration,shoulder_left_angle,shoulder_left_angular_velocity,shoulder_left_angular_acceleration,shoulder_right_angle,...,spine_angular_acceleration,ankle_left_angle,ankle_left_angular_velocity,ankle_left_angular_acceleration,ankle_right_angle,ankle_right_angular_velocity,ankle_right_angular_acceleration,center_distance,center_speed,Label
0,20.680788,-12.271715,184.552498,117.980466,2498.058166,7052.299677,25.160310,365.426342,4574.281863,43.718651,...,-2055.415069,121.146172,-1041.416422,-5435.284408,134.870221,-1445.213398,-8822.845456,1.942890e-16,1.165734e-14,0.0
1,25.160022,6.151750,-3977.881772,126.860540,235.076656,-74874.669237,32.936111,152.476062,-12993.440283,49.397831,...,-6805.643321,107.223897,-181.176147,40327.731608,116.830968,-294.094849,53051.507406,3.955170e-16,2.373102e-14,0.0
2,20.885847,-144.867774,-425.943202,125.816355,2.235858,-8595.416016,30.242845,-67.688334,-5016.894535,50.575347,...,3083.921846,115.106967,302.841298,7890.131248,125.067060,323.170182,12170.580699,4.437638e-16,2.662583e-14,0.0
3,20.331096,-8.046357,4197.226968,126.935069,-51.437211,1911.131293,30.679833,-14.753756,-542.685302,53.972380,...,10712.091864,117.318607,81.828228,-9271.617768,127.603307,111.591175,-7018.747461,2.087443e-16,1.252466e-14,0.0
4,20.617635,-4.960209,-323.045662,124.101781,65.940234,8833.618161,29.751053,-85.777844,-941.782118,48.057296,...,-438.093245,117.834575,-6.212628,-5679.841082,128.786766,89.211933,-2096.623552,2.498002e-16,1.498801e-14,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296562,25.626388,31.437188,-739.170751,116.499138,-91.135295,1752.576522,14.011173,-2.501825,-726.467353,50.954784,...,-73.892263,116.095380,-24.295745,454.977069,107.489540,-32.471702,-799.993634,4.996004e-16,1.448841e-14,1.0
296563,26.102950,-14.325184,-1129.321421,114.687117,27.905029,3241.283093,13.426289,-28.977733,-756.436352,51.931975,...,11.409774,115.623805,14.651826,1599.422870,106.294996,-32.522526,696.040891,2.498002e-16,7.244205e-15,1.0
296564,24.638445,-46.447048,53.555766,118.423623,132.401470,-598.446070,12.012709,-54.669849,1400.294337,51.446722,...,-529.589874,117.105850,86.009281,375.720456,105.246607,15.531118,633.252007,3.532708e-16,1.024485e-14,1.0
296565,22.899705,-10.631683,7443.498688,123.818253,-13.367114,-12608.788308,9.655955,67.594290,6448.057128,58.201012,...,-9117.846468,121.555480,40.563581,-2798.553521,107.366108,11.150026,-79.421324,3.532708e-16,1.024485e-14,1.0


In [ ]:
# X y
X = df.drop(columns='Label')
y = df['Label']

In [ ]:
# Train/Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 정규화
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 다시 DataFrame으로 변환 (컬럼명 유지)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

In [ ]:
# SMOTE : class 불균형 해소를 위한 합성 샘플 생성
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)  # ✅ y → y_train

print("원본 데이터 분포:\n", y.value_counts())
print("SMOTE 적용 후 데이터 분포:\n", y_train_res.value_counts())

원본 데이터 분포:
 Label
1.0    194917
0.0    101650
Name: count, dtype: int64
SMOTE 적용 후 데이터 분포:
 Label
1.0    155933
0.0    155933
Name: count, dtype: int64


In [ ]:
def objective(trial):
    # 정수형 파라미터를 직접 지정할 수 있어 편리합니다.
    max_depth = trial.suggest_int('max_depth', 5, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 50, 200)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 10, 100)
    ccp_alpha = trial.suggest_float('ccp_alpha', 0.0, 0.001)              # 가지치기로 일반화 성능 향상

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        ccp_alpha=ccp_alpha,
        random_state=42,
        criterion='gini',
        class_weight='balanced'
    )

    score = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='recall').mean()
    return score


# 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, callbacks=[optuna.study.MaxTrialsCallback(100, states=(optuna.trial.TrialState.COMPLETE,))])

print(f"최고 Recall 점수 : {study.best_value}")

[I 2026-05-12 12:12:32,873] A new study created in memory with name: no-name-2fde4ff9-15c2-4efc-8aa9-11eddd65ae1e
[I 2026-05-12 12:16:02,035] Trial 0 finished with value: 0.9194781233831767 and parameters: {'max_depth': 11, 'min_samples_split': 163, 'min_samples_leaf': 38, 'ccp_alpha': 0.00014043461314907903}. Best is trial 0 with value: 0.9194781233831767.
[I 2026-05-12 12:18:58,217] Trial 1 finished with value: 0.9183815769795782 and parameters: {'max_depth': 9, 'min_samples_split': 152, 'min_samples_leaf': 50, 'ccp_alpha': 8.833248402372207e-06}. Best is trial 0 with value: 0.9194781233831767.
[I 2026-05-12 12:22:58,241] Trial 2 finished with value: 0.949657826589615 and parameters: {'max_depth': 14, 'min_samples_split': 194, 'min_samples_leaf': 12, 'ccp_alpha': 0.0007803203721975753}. Best is trial 2 with value: 0.949657826589615.
[I 2026-05-12 12:24:46,837] Trial 3 finished with value: 0.9377232661630724 and parameters: {'max_depth': 5, 'min_samples_split': 159, 'min_samples_leaf'

최고 Recall 점수 : 0.9522872074573577


In [ ]:
# 가장 좋았던 파라미터로 최종 모델 생성
final_clf = DecisionTreeClassifier(**study.best_params)
final_clf.fit(X_train_res, y_train_res)

y_pred_dt = final_clf.predict(X_test_scaled)

print("Decision Tree 정확도:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Decision Tree Classification Report:\n", classification_report(y_test, y_pred_dt))

Decision Tree 정확도: 0.8759820615706241
Decision Tree Confusion Matrix:
 [[14766  5564]
 [ 1792 37192]]
Decision Tree Classification Report:
               precision    recall  f1-score   support

         0.0       0.89      0.73      0.80     20330
         1.0       0.87      0.95      0.91     38984

    accuracy                           0.88     59314
   macro avg       0.88      0.84      0.86     59314
weighted avg       0.88      0.88      0.87     59314



In [ ]:
print(f"**study.best_params : {study.best_params}")

**study.best_params : {'max_depth': 15, 'min_samples_split': 191, 'min_samples_leaf': 43, 'ccp_alpha': 0.0008993649522689042}
